# Assignment 4: Implement a Contrastive Learning Method from Description (100 points)

In this assignment, you will implement a contrastive learning method given a paper-style description. This tests your ability to translate a research description into working PyTorch code.

## Background: Supervised Contrastive Learning (SupCon)

Standard contrastive learning (e.g., SimCLR) uses data augmentation to create positive pairs and treats all other samples as negatives. **Supervised Contrastive Learning** (Khosla et al., 2020) extends this by using label information: samples with the **same label** are positives, and samples with **different labels** are negatives.

### Method Description

Given a batch of $N$ samples $\{(x_i, y_i)\}_{i=1}^{N}$ where $y_i$ is the class label:

1. **Augmentation:** Create two views of each sample: $\tilde{x}_{2i-1}$ and $\tilde{x}_{2i}$ from $x_i$. This gives $2N$ augmented samples.

2. **Encoding:** Pass all $2N$ augmented samples through an encoder $f(\cdot)$ to get representations $h_i = f(\tilde{x}_i) \in \mathbb{R}^d$.

3. **Projection:** Pass through a projection head $g(\cdot)$ to get $z_i = g(h_i) \in \mathbb{R}^{d_p}$. Normalize: $z_i \leftarrow z_i / \|z_i\|_2$.

4. **SupCon Loss:** For each anchor $i$ in the batch of $2N$ samples:

$$\mathcal{L}_i = \frac{-1}{|P(i)|} \sum_{p \in P(i)} \log \frac{\exp(z_i \cdot z_p / \tau)}{\sum_{a \in A(i)} \exp(z_i \cdot z_a / \tau)}$$

Where:
- $A(i) = \{1, \ldots, 2N\} \setminus \{i\}$ (all indices except $i$)
- $P(i) = \{p \in A(i) : \tilde{y}_p = \tilde{y}_i\}$ (positive set: same label, excluding self)
- $\tau > 0$ is the temperature parameter

5. **Total loss:** $\mathcal{L} = \frac{1}{2N} \sum_{i=1}^{2N} \mathcal{L}_i$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import math

---

> **WARNING:** Do not modify any code outside of the designated solution areas.

---

## Part 1: Understanding the Loss (10 points)

**[Non-coding]**

1. (3 points) In a batch of $N = 4$ samples with labels $[0, 0, 1, 1]$, after augmentation we have $2N = 8$ samples with labels $[0, 0, 0, 0, 1, 1, 1, 1]$. For anchor $i = 0$ (label 0), what are $A(0)$ and $P(0)$? How many terms are in each set?

2. (3 points) How does SupCon loss differ from SimCLR loss? What changes when we go from self-supervised to supervised contrastive learning?

3. (2 points) What role does the temperature $\tau$ play? What happens as $\tau \to 0$ and $\tau \to \infty$?

4. (2 points) Why do we normalize the projection vectors $z_i$ to unit length?

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 2: Encoder and Projection Head (10 points)

**[Coding]** Implement the encoder and projection head for a simple setting (1D feature vectors).

**Encoder $f$:**
- Input: $(B, d_{\text{in}})$
- Architecture: Linear($d_{\text{in}}$, 256) → ReLU → Linear(256, 128) → ReLU → Linear(128, $d$)
- Output: $(B, d)$

**Projection head $g$:**
- Input: $(B, d)$
- Architecture: Linear($d$, $d$) → ReLU → Linear($d$, $d_p$)
- Output: $(B, d_p)$, L2-normalized

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class Encoder(nn.Module):
    def __init__(self, d_in, d_out=64):
        super().__init__()
        pass  # YOUR CODE

    def forward(self, x):
        pass  # YOUR CODE


class ProjectionHead(nn.Module):
    def __init__(self, d_in, d_proj=32):
        super().__init__()
        pass  # YOUR CODE

    def forward(self, h):
        # Returns L2-normalized projections
        pass  # YOUR CODE

""" END OF THIS PART """

## Part 3: SupCon Loss Implementation (20 points)

**[Coding]** Implement the Supervised Contrastive Loss.

This is the core of the assignment. The function takes:
- `features`: $(2N, d_p)$ — L2-normalized projection vectors for all augmented samples
- `labels`: $(2N,)$ — the class label for each augmented sample
- `temperature`: $\tau$ (default 0.07)

**Implementation strategy (vectorized):**
1. Compute similarity matrix: $S = z z^T / \tau$ of shape $(2N, 2N)$
2. Create mask for positives: `mask[i, j] = 1` if `labels[i] == labels[j]` and `i != j`
3. For numerical stability: subtract `max(S)` from each row before exp (log-sum-exp trick)
4. Compute log-softmax over the denominator $A(i)$
5. Mask out the positive entries and average

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def supcon_loss(features, labels, temperature=0.07):
    """
    Supervised Contrastive Loss.
    
    Args:
        features: (2N, d_p) L2-normalized projection vectors
        labels: (2N,) integer class labels
        temperature: scalar tau
    
    Returns:
        loss: scalar
    """
    pass  # YOUR CODE

""" END OF THIS PART """

## Part 4: Loss Verification (8 points)

**[Coding]** Verify your loss implementation with known cases.

1. (4 points) Create a test case where all features in the same class are identical and features in different classes are orthogonal. The loss should be low.

2. (4 points) Create a test case where all features are random. The loss should be higher. Print both losses and verify the relationship.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Test case 1: well-separated features
# Test case 2: random features

""" END OF THIS PART """

## Part 5: Data Augmentation (8 points)

**[Coding]** For our simple 1D setting, implement augmentation by adding Gaussian noise.

Given a batch $(x, y)$ of shape $((B, d_{\text{in}}), (B,))$, create two augmented views:
- $\tilde{x}_1 = x + \epsilon_1$ where $\epsilon_1 \sim \mathcal{N}(0, \sigma^2 I)$
- $\tilde{x}_2 = x + \epsilon_2$ where $\epsilon_2 \sim \mathcal{N}(0, \sigma^2 I)$

Return concatenated views and labels: $((2B, d_{\text{in}}), (2B,))$

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def augment_batch(x, y, sigma=0.1):
    """
    Create two augmented views of the batch.
    
    Args:
        x: (B, d_in)
        y: (B,)
        sigma: noise standard deviation
    
    Returns:
        x_aug: (2B, d_in)
        y_aug: (2B,)
    """
    pass  # YOUR CODE

""" END OF THIS PART """

## Part 6: Full SupCon Model (10 points)

**[Coding]** Combine encoder and projection head into a full SupCon model.

The model should have two modes:
- **Contrastive mode:** Returns normalized projections $z$ (for computing SupCon loss during pretraining)
- **Linear evaluation mode:** Returns encoder features $h$ (for training a linear classifier after pretraining)

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class SupConModel(nn.Module):
    def __init__(self, d_in, d_enc=64, d_proj=32):
        super().__init__()
        pass  # YOUR CODE

    def forward(self, x, return_projections=True):
        """
        Args:
            x: (B, d_in)
            return_projections: if True, return z (for contrastive loss)
                                if False, return h (for linear eval)
        """
        pass  # YOUR CODE

""" END OF THIS PART """

## Part 7: Synthetic Dataset (6 points)

**[Coding]** Create a synthetic classification dataset.

- 5 classes, each a Gaussian cluster in $\mathbb{R}^{20}$
- Class $k$ has mean $\mu_k$ sampled uniformly from $[-5, 5]^{20}$ (use a fixed seed)
- Standard deviation $\sigma = 1.0$ per class
- 200 samples per class → 1000 total
- Split: 800 train, 200 test

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Generate synthetic dataset

""" END OF THIS PART """

## Part 8: Contrastive Pretraining (12 points)

**[Coding]** Train the SupCon model with contrastive pretraining.

- 50 epochs, Adam optimizer, lr=1e-3
- Batch size 64
- For each batch: augment, encode, project, compute SupCon loss
- Print loss every 10 epochs

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Contrastive pretraining loop

""" END OF THIS PART """

## Part 9: Linear Evaluation (10 points)

**[Coding]** Evaluate the learned representations.

1. Freeze the encoder (no gradients)
2. Train a linear classifier on the encoder features: Linear($d$, num_classes)
3. Train for 50 epochs with Adam, lr=1e-2
4. Report test accuracy

Also train a baseline: the same linear classifier directly on the raw input features (no encoder). Compare the two accuracies.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Linear evaluation: SupCon features vs. raw features

""" END OF THIS PART """

## Part 10: Analysis (6 points)

**[Non-coding]**

1. (2 points) Why do we train a linear classifier on top of the frozen encoder, rather than fine-tuning the whole model? What does this evaluation protocol measure?

2. (2 points) The projection head $g$ is discarded after pretraining. Why is it needed during training but not for downstream tasks?

3. (2 points) How would you adapt this method for a setting where some samples have labels and others do not (semi-supervised learning)?

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """